**First hands-on ADK agent.** This notebook builds a minimal Gemini-backed agent using Google's Agent Development Kit (ADK): a search-enabled `Agent` wrapped in `Gemini`, run through `InMemoryRunner`, with HTTP retry config for resilience. It also shows the `adk` CLI for scaffolding a project and launching a local web UI.

In [ ]:
pip install google-adk

`google-adk` is Google's framework for building, testing, and deploying Gemini-backed agents — it bundles the `Agent`/`Runner` abstractions, built-in tools (like `google_search`), and a CLI (`adk`) for scaffolding and local dev servers.

In [ ]:
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

print("✅ ADK components imported successfully.")

- `Agent` — the core object combining a model, instructions, and tools.
- `Gemini` — wraps a specific Gemini model (and lets you attach things like retry options).
- `InMemoryRunner` — orchestrates a session/conversation with an agent, all in memory (no persistent storage).
- `google_search` — a built-in tool the agent can call to search the web.
- `types` — shared request/response types (used below for `HttpRetryOptions`).

In [ ]:
from dotenv import load_dotenv

# Load variables from .env into the environment
load_dotenv()

`load_dotenv()` reads key/value pairs from a `.env` file in the working directory into `os.environ` — this is how `GOOGLE_API_KEY` gets picked up below without hardcoding it in the notebook.

In [ ]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

`HttpRetryOptions` configures automatic retries for transient API failures — up to 5 attempts with exponential backoff (`exp_base=7`), only for the listed status codes (rate-limiting and server errors). Passing this into `Gemini(...)` makes the agent resilient to flaky network/API conditions without extra try/except code.

In [ ]:
root_agent = Agent(
    name="helpful_assistant",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="A simple agent that can answer general questions.",
    instruction="You are a helpful assistant. Use Google Search for current info or if unsure.",
    tools=[google_search],
)

print("✅ Root Agent defined.")

`root_agent` ties it together: a `Gemini` model (with the retry config attached), a natural-language `instruction` steering its behavior, and `tools=[google_search]` — giving the model the ability to issue live web searches when its own knowledge isn't enough, rather than answering only from training data.

In [ ]:
runner = InMemoryRunner(agent=root_agent)
# Runner act as a orchestration 
print("✅ Runner created.")

`InMemoryRunner` manages the conversation loop — sessions, message history, and tool-call execution — for a given agent, keeping everything in memory (nothing persists once the process ends). It's the lightweight way to run an agent locally before wiring up a real backend.

In [ ]:
# before run this we should configure the api key like below in a file named .env same directory of out file 
## echo 'GOOGLE_API_KEY="REMOVED"' > .env

response = await runner.run_debug(
    "what is name of new NewYork mayor and until when? then give me history of this person"
)

`run_debug` is a convenience method for interactive/notebook use: it runs a turn and pretty-prints the session transcript (as seen above), unlike a bare `run`/`run_async` call which just returns structured events for you to handle yourself. Note: the `.env` file itself should never be committed — keep real API keys out of version control, even in the comment that creates it.

In [ ]:
# TO create google ADK web UI in windows Power shell run 7
## adk create sample-agent --model gemini-2.5-flash-lite

# To run this interface 
## adk web .

The `adk` CLI is separate from the Python API used above: `adk create` scaffolds a new agent project (folder, config, boilerplate) from a template, and `adk web .` launches a local web UI for chatting with and debugging the agent(s) in the current directory — useful for iterating without writing a runner script each time.